# Intent Based Agentic AI Implementation

### ReActXen setup

The following cell adds the ReActXen source directory to `sys.path` and provides a small verification of core `reactxen` modules. Run the next code cell to actually perform the setup and check imports.

In [31]:
import os
import sys
from pathlib import Path

try:
    # Get the absolute path of the current notebook directory
    notebook_dir = Path(__file__).resolve().parent
    workspace_root = notebook_dir.parent
    reactxen_path = workspace_root / "ReActXen" / "src" 
    print(f"✅ Method 1 - Using __file__: {reactxen_path}")
except NameError:
    # Method 2: For Jupyter notebooks (__file__ not available)
    notebook_dir = Path.cwd()
    workspace_root = notebook_dir.parent
    reactxen_path = workspace_root / "ReActXen" / "src" 
    print(f"✅ Method 2 - Using Path.cwd(): {reactxen_path}")

# Verify the path exists
if reactxen_path.exists():
    print(f"✅ ReActXen path exists: {reactxen_path}")
    # Add to sys.path
    sys.path.insert(0, str(reactxen_path))
    print(f"✅ Added to sys.path: {reactxen_path}")
else:
    print(f"❌ ReActXen path does not exist: {reactxen_path}")
    print("Please check the directory structure.")

# Test the import
try:
    # click on imported function to check if function can be accessed
    from ReActXen.src.reactxen.prebuilt.create_reactxen_agent import create_reactxen_agent

    print("✅ Successfully imported create_reactxen_agent")
except ImportError as e:
    print(f"❌ Import failed: {e}")     # TODO : ignore this, as I just needed access to the functions internally
    # print("Available paths:")
    # for i, p in enumerate(sys.path[:5]):
    #     print(f"  {i}: {p}")

✅ Method 2 - Using Path.cwd(): /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
✅ ReActXen path exists: /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
✅ Added to sys.path: /Users/ayandas/Desktop/research_ibm/Intent-Based-Industrial-Automation/ReActXen/src
❌ Import failed: No module named 'ReActXen'


In [43]:
# install neccessary packages
%pip install --upgrade pip
%pip install langchain-ibm langchain-community haversine pandas langchain-ibm langchain-community haversine pandas ibm-watsonx-ai numpy xgboost

Could not fetch URL https://pypi.org/simple/pip/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pip/ (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)'))) - skipping

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Could not fetch URL https://pypi.org/simple/xgboost/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/xgboost/ (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)'))) - skipping
ERROR: Could not find a version that satisfies the requirement xgboost (from versions: none)
ERROR: No matching distribution found for xgboost

[notice

In [36]:
# Handle necessary installation and setup
# NOTE : use this code block for any/all imports related to subseuquent code logic of agentic implementation

import argparse
# from ReActXen.src.reactxen.prebuilt.create_reactxen_agent import create_reactxen_agent
import json
from getpass import getpass
import warnings
import pandas as pd
import langchain
from langchain_ibm import WatsonxLLM
from langchain_core.prompts import PromptTemplate
from ibm_watsonx_ai import APIClient


# filter out all warnings
warnings.filterwarnings("ignore")

In [39]:
'''define additional parameters that create_reactxen_agent will accept'''

# sample experiment to check how PromptTemplate works
# 2 methods available
# prompt = PromptTemplate.from_template("Absolute {Absolute_Universe_Hero}")

# # returns updated prompt, but doesn't mutate, meaning value is cloned.
# formatted_prompt = prompt.format(Absolute_Universe_Hero="Batman")
# print(formatted_prompt)

agent_prompt = PromptTemplate(
    input_variables=["train_data","test_data","ground_truth"],
    template=(
        "You are an AI Assistant specialized in predictive, corrective, preventative and reliabillity centered maintenance",
        "\nYou have been given three sets of datasets and the breakdown of each of them is as follows:\n\n"
        "{train_data} : Contains complete run-to-failure trajectories for 100 engines. Each row represents one time step (cycle) of an engine. Each engine runs from a healthy state (cycle 1) to its simulated failure cycle\n"
        "{test_data} : Contains partial trajectories involving another 100 engines, with each engine stopping right before failure. Which means the full lifetime cannot be seen. This data can be used to predict how many more remaining useful lifecycles (RUL for short) remains\n"
        "{ground_truth} : Contains 100 numbers, one per test engine, and contains the true number of remaining useful cycles after the last record of the engine's trajectory. This is the 'answer' key to be used for evaluation"
    )
)


# There are 2 methods involved when it comes to calculating RUL
#
# This is something to improve the thinking process of agents
# 1. Linear RUL
# 2. Piecewise RUL
reflect_prompt = PromptTemplate(
    input_variables=["predicted_val", "actual_val"],
    template=(
        # This part of the prompt will be useful for prediction of rul
        # This is a binary scenario
        "In the event that {predicted_val} != {actual_val} using the tools you have been provided, reflect on why this occured and provide a single liner justification. Then try again, don't just look at the final answer, understand in-depth the steps needed to be taken to arrive at the final answer."
        "In the event that {predicted_val} == {actual_val} using the tools you have been provided, provide a brief explanation of which machine learning technique was used to derive at this answer, and whether you have tested with other models as well. The explanation should be factual, and should match a typical SME knowledge."
        "In addition to that, consider which amongst the list of prediction techniques has proven to yield the highest accuracy of prediction that matches the actual value"
        "In the event that the problem is more open-ended, perform in-depth root cause analysis and derive a simple explanation justifying the response that has been provided, reference the neccessary data/facts as part of the justification."
    ),
)

Absolute Batman


@misc{Sahoo_Data-Driven_Remaining_Useful_2020,
author = {Sahoo, Biswajit},
doi = {10.5281/zenodo.5890595},
month = {9},
title = {Data-Driven Remaining Useful Life (RUL) Prediction},
url = {https://biswajitsahoo1111.github.io/rul_codes_open/},
year = {2020}
}

In [6]:
# TODO : list of tools to implement and how agent should interact
'''
an agent that has access to this info (this will be a subcomponent of overall flow)

1. get_reference_train_data
2. get_reference_test_data
3. get_rul_model
4. train_rul_model
5. predict_rul_model
'''

# potential utterance we are working with
sample_utterace =  "We should focus on the engines that are running on fumes, which ones are likely to give out in the next 20 cycles? Provide me a list of engine ids."

In [ ]:
def get_reference_train_data():
    '''fairly straightforward, retrieves the train data and for mats it using pandas'''
    pass

def get_reference_test_data():
    '''fairly straightforward, retrieves the test data and formats it using pandas'''
    pass

def get_rul_model():
    '''
    simply a getter function (might be a bit redundant unless series of training and testing model logic is present)
    '''
    pass

def train_rul_model():
    '''
    sub-agent that can call on the functions responsible for running the training on the models
    
    smallest_subproblem : include the logic for training of at least one technique
    '''
    pass

# sub-agent that will have access to various methods of prediction via tool
def predict_rul_agent():
    '''
    It's tools will involve using the functions for predicting RUL based on the models that has been trained
    
    smallest_subproblem : include the logic for prediction of at least one techniques
    '''
    pass